# Download the CMEMS currents

Run this once. It fetches the CMEMS subset that the three Cabo Verde notebooks
open from disk and writes it to `data/cabo_verde_currents_hourly.nc`. Re-running
it is cheap: if the file is already there and opens, nothing is downloaded.

`examples/data/` is gitignored, so the file is never committed and a fresh
clone has to run this notebook before the three Cabo Verde ones.

This assumes
[`copernicusmarine`](https://help.marine.copernicus.eu/en/collections/4060068-copernicus-marine-toolbox)
has credentials.

In [1]:
from pathlib import Path

import copernicusmarine as cm
import xarray as xr

/Users/wrath/src/github.com/geomar-od-lagrange/lcs_parcels/.pixi/envs/examples/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## The subset

Hourly surface velocity (`uo`, `vo`) from Global Ocean Physics Analysis and
Forecast, `GLOBAL_ANALYSISFORECAST_PHY_001_024`,
[doi:10.48670/moi-00016](https://doi.org/10.48670/moi-00016), dataset
`cmems_mod_glo_phy_anfc_0.083deg_PT1H-m`. A box around Cabo Verde, first depth
level only. The examples release particles between 2025-08-01 and 2025-08-11
inside a smaller box; the extra day at each end and the margin in longitude and
latitude are there so trajectories stay inside the data.

In [2]:
target = Path("data/cabo_verde_currents_hourly.nc")
target.parent.mkdir(parents=True, exist_ok=True)

In [3]:
try:
    xr.open_dataset(target).close()
    have_file = True
except (FileNotFoundError, OSError):
    have_file = False

print(
    f"{target}: already here, skipping the download"
    if have_file
    else f"{target}: missing, downloading it"
)

data/cabo_verde_currents_hourly.nc: already here, skipping the download


In [4]:
if not have_file:
    ds = cm.open_dataset(
        dataset_id="cmems_mod_glo_phy_anfc_0.083deg_PT1H-m",
        variables=["uo", "vo"],
        minimum_longitude=-30.5,
        maximum_longitude=-17.5,
        minimum_latitude=10.0,
        maximum_latitude=22.0,
        minimum_depth=0.0,
        maximum_depth=1.0,
        start_datetime="2025-07-31",
        end_datetime="2025-08-12",
    ).load()
    ds.to_netcdf(target)

## What landed on disk

In [5]:
currents = xr.open_dataset(target)
currents

<xarray.Dataset> Size: 53MB
Dimensions:    (time: 289, depth: 1, latitude: 145, longitude: 157)
Coordinates:
  * time       (time) datetime64[ns] 2kB 2025-07-31 ... 2025-08-12
  * depth      (depth) float32 4B 0.494
  * latitude   (latitude) float32 580B 10.0 10.08 10.17 ... 21.83 21.92 22.0
  * longitude  (longitude) float32 628B -30.5 -30.42 -30.33 ... -17.58 -17.5
Data variables:
    uo         (time, depth, latitude, longitude) float32 26MB ...
    vo         (time, depth, latitude, longitude) float32 26MB ...
Attributes:
    Conventions:               CF-1.8
    area:                      Global
    contact:                   https://marine.copernicus.eu/contact
    credit:                    E.U. Copernicus Marine Service Information (CM...
    institution:               Mercator Ocean International
    licence:                   http://marine.copernicus.eu/services-portfolio...
    producer:                  CMEMS - Global Monitoring and Forecasting Centre
    references:                http://marine.copernicus.eu
    source:                    MOI GLO12
    title:                     hourly mean fields from Global Ocean Physics A...
    copernicusmarine_version:  2.4.1